Kütüphane importları


In [12]:
!pip install pyroomacoustics
from google.colab import drive
import pyroomacoustics as pra
import pandas as pd
import numpy as np
import torchaudio
import zipfile
import ast
import os

Önce veri setini çekip ihtiyacımız olan csv dosyasını alalım

In [13]:
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/dataset.zip"
extract_path = "/content/dataset/"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP başarıyla çıkarıldı")

test_csv = '/content/drive/MyDrive/SSL_Projesi/test_metadata.csv'
df = pd.read_csv(test_csv)

print(df.head())
print("Toplam test örneği:", len(df))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ZIP başarıyla çıkarıldı
                                          audio_path  azimuth_deg  distance_m  \
0  /content/dataset/dataset/session_1775030812_5c...    79.932860    2.810450   
1  /content/dataset/dataset/session_1775040605_e2...   262.116227    4.355966   
2  /content/dataset/dataset/session_1775040766_d2...    70.257374    3.220559   
3  /content/dataset/dataset/session_1775030073_23...   234.963847    3.535452   
4  /content/dataset/dataset/session_1775041687_e1...   259.779367    2.567600   

      pos_x     pos_y  pos_z  snr_db  ambient_noise_db  rt60 room_dimension  
0  2.991272  4.900000    1.5      30                15   0.3      (5, 5, 3)  
1  4.402518  0.685205    2.5       0                 0   0.3    (10, 10, 5)  
2  3.587891  4.900000    1.5      30                 0   0.5      (5, 5, 3)  
3  0.470321  0.100000    1.5      15            

Mikrofon Pozisyonları

In [14]:
radius = 0.1

mics = []

for i in range(7):
    angle = 2 * np.pi * i / 7
    x = radius * np.cos(angle)
    y = radius * np.sin(angle)
    mics.append([x, y, 0])

mics.append([0, 0, 0])

mic_locs = np.array(mics).T

In [15]:
def srp_phat(signals, fs, nfft, mic_locs):

  localizer = pra.doa.SRP(
    mic_locs,
    fs=fs,
    nfft=nfft,
    c=343.0,
    num_src=1
  )

  localizer.locate_sources(signals)

  spatial_spectrum = localizer.grid.values
  grid_angles = np.degrees(localizer.grid.azimuth)

  # 3 değeri de döndür
  return float(np.degrees(localizer.azimuth_recon[0]) % 360), spatial_spectrum, grid_angles

In [18]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

def save_combined_spectrum_plot(
    grid_angles,
    spec_success, true_success, est_success,
    spec_fail, true_fail, est_fail,
    filename="/content/music_karsilastirma.png"
):
    # 1. MAKALE STANDARTLARI İÇİN SEABORN AYARLARI
    sns.set_theme(style="ticks", context="paper", font_scale=1.2)
    plt.rcParams["font.family"] = "serif"

# 1 satır, 2 sütunlu geniş bir figür oluştur
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

   # --- Yeni Akademik Renk Paleti ---
    color_spectrum = "#003366"  # Tok Akademik Lacivert
    color_true = "#d7191c"      # Dikkat çekici Kırmızı
    color_est = "#000000"       # Siyah (Kesik çizgilerde en iyi kontrast)

    # ==========================================
    # (a) BAŞARILI SENARYO (Sol Grafik)
    # ==========================================
    ax1 = axes[0]
    spec_success_norm = spec_success / np.max(spec_success)

    sns.lineplot(x=grid_angles, y=spec_success_norm, ax=ax1,
                label="Psödo-Spektrum $P(\\theta)$", color=color_spectrum, linewidth=2)
    ax1.axvline(x=true_success, color=color_true, linestyle='--', linewidth=2.5,
                label=f"Gerçek Açı ({true_success:.1f}°)")
    ax1.axvline(x=est_success, color=color_est, linestyle=':', linewidth=2.5,
                label=f"Tahmin ({est_success:.1f}°)")

    ax1.set_xlim([0, 360])
    ax1.set_ylim([0, 1.05])
    ax1.set_xlabel("Arama Açısı (Derece)", fontweight='bold')
    ax1.set_ylabel("Normalize Psödo-Spektrum", fontweight='bold')
    ax1.set_title("(a) Başarılı Kestirim", pad=15, fontweight='bold')

    ax1.legend(loc="lower right", frameon=True, edgecolor='black')
    ax1.grid(axis='y', linestyle=':', alpha=0.6)

    # ==========================================
    # (b) HATALI SENARYO (Sağ Grafik)
    # ==========================================
    ax2 = axes[1]
    spec_fail_norm = spec_fail / np.max(spec_fail)

    sns.lineplot(x=grid_angles, y=spec_fail_norm, ax=ax2,
                label="Psödo-Spektrum $P(\\theta)$", color=color_spectrum, linewidth=2)
    ax2.axvline(x=true_fail, color=color_true, linestyle='--', linewidth=2.5,
                label=f"Gerçek Açı ({true_fail:.1f}°)")
    ax2.axvline(x=est_fail, color=color_est, linestyle=':', linewidth=2.5,
                label=f"Tahmin ({est_fail:.1f}°)")

    ax2.set_xlim([0, 360])
    ax2.set_ylim([0, 1.05])
    ax2.set_xlabel("Arama Açısı (Derece)", fontweight='bold')
    ax2.set_ylabel("Normalize Psödo-Spektrum", fontweight='bold')
    ax2.set_title("(b) Hatalı Kestirim", pad=15, fontweight='bold')

    ax2.legend(loc="lower right", frameon=True, edgecolor='black')
    ax2.grid(axis='y', linestyle=':', alpha=0.6)

    # ==========================================
    # SON RÖTUŞLAR VE KAYDETME
    # ==========================================
    sns.despine(fig=fig, top=True, right=True)
    plt.tight_layout()

    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.close()
    print(f"✅ Karşılaştırmalı grafik başarıyla kaydedildi: {filename}")

Algoritmayı csvdeki verilerle çalıştırmak

In [19]:
import os
import ast
import pandas as pd
import numpy as np
import torchaudio
import pyroomacoustics as pra

# ==========================================
# 1. HAZIRLIK VE GÜVENLİ FİLTRELEME
# ==========================================
results = []
plot_data = []
plots_dir = "/content/"
nfft = 1024
hop = nfft // 2

# Sütun isimlerini dinamik yakala
col_file = "audio_path" if "audio_path" in df.columns else "file"
col_true = "azimuth_deg" if "azimuth_deg" in df.columns else "true_angle"

# Test edilecek hedef dosyalar
hedef_kelimeler = r"1775030993_77e7d1/sample_00002\.wav|1775041585_a67946/sample_00002\.wav"
test_df = df[df[col_file].str.contains(hedef_kelimeler, na=False, regex=True)].reset_index(drop=True)

# Çökme hatasını engelleyen saf Python sözlüğüne (List of Dicts) çevirme
records = test_df.to_dict(orient="records")
print(f"SRP-PHAT İçin İşlenecek Dosya Sayısı: {len(records)}")

# ==========================================
# 2. ÇÖKMEYEN İŞLEM DÖNGÜSÜ
# ==========================================
for i, row in enumerate(records):
    rel_path = row[col_file]
    true_angle = row[col_true]

    wav_path = os.path.join(extract_path, rel_path)
    mic_locs[2, :] = ast.literal_eval(row["room_dimension"])[2] / 2

    # Sesi yükle
    waveform, sr = torchaudio.load(wav_path)
    waveform = waveform.numpy()

    # SRP-PHAT'a özel STFT analizi
    X = pra.transform.stft.analysis(
        waveform.T,
        L=nfft,
        hop=hop
    ).transpose(2, 1, 0)

    # SRP-PHAT Algoritmasını Çalıştır
    estimated_angle, spatial_spectrum, grid_angles = srp_phat(X, sr, nfft, mic_locs)

    # Dairesel Hata Hesaplama (0-360 döngüsü)
    raw_error = abs(true_angle - estimated_angle)
    error = min(raw_error, 360 - raw_error)

    results.append({
        "method": "srp-phat",
        "file": rel_path,
        "true_angle": true_angle,
        "estimated_angle": estimated_angle,
        "angular_error": error
    })

    plot_data.append({
        "spec": spatial_spectrum,
        "true": true_angle,
        "est": estimated_angle,
        "error": error
    })

    print(f"[{i+1}/2] İşlendi: {rel_path.split('/')[-1]} | Hata: {error:.2f}°")

# ==========================================
# 3. GRAFİK ÇİZİMİNİ TETİKLEME
# ==========================================
if len(plot_data) == 2:
    if plot_data[0]["error"] < plot_data[1]["error"]:
        idx_success, idx_fail = 0, 1
    else:
        idx_success, idx_fail = 1, 0

    save_combined_spectrum_plot(
        grid_angles=grid_angles,
        spec_success=plot_data[idx_success]["spec"],
        true_success=plot_data[idx_success]["true"],
        est_success=plot_data[idx_success]["est"],
        spec_fail=plot_data[idx_fail]["spec"],
        true_fail=plot_data[idx_fail]["true"],
        est_fail=plot_data[idx_fail]["est"],
        filename=os.path.join(plots_dir, "srp_phat_karsilastirma.png")
    )

# Sonuçları Kaydet
results_df = pd.DataFrame(results)
results_df.to_csv("srp-phat-results.csv", index=False)
print("✅ İşlemler tamamlandı, CSV ve SRP-PHAT grafiği kaydedildi.")

SRP-PHAT İçin İşlenecek Dosya Sayısı: 2
[1/2] İşlendi: sample_00002.wav | Hata: 151.67°
[2/2] İşlendi: sample_00002.wav | Hata: 0.00°
✅ Karşılaştırmalı grafik başarıyla kaydedildi: /content/srp_phat_karsilastirma.png
✅ İşlemler tamamlandı, CSV ve SRP-PHAT grafiği kaydedildi.


Hesaplamalardan sonra rapor için gerekli verileri alalım